In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/melb_data.csv")

In [3]:
df.shape
df.head()
df.info()
df.isnull().sum().sort_values(ascending=False)

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      13580 non-null  float64
 18  Longtitude     13

BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Car                62
Suburb              0
Price               0
Type                0
Rooms               0
Address             0
Distance            0
Method              0
Date                0
SellerG             0
Bathroom            0
Bedroom2            0
Postcode            0
Landsize            0
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

In [4]:
drop_cols = [
    "BuildingArea",
    "YearBuilt",
    "CouncilArea",
    "Address",
    "Date",
    "SellerG"
]

df = df.drop(columns=drop_cols)

df["Car"] = df["Car"].fillna(df["Car"].median())

df.isnull().sum()

Suburb           0
Rooms            0
Type             0
Price            0
Method           0
Distance         0
Postcode         0
Bedroom2         0
Bathroom         0
Car              0
Landsize         0
Lattitude        0
Longtitude       0
Regionname       0
Propertycount    0
dtype: int64

In [5]:
df.shape
df.dtypes

Suburb               str
Rooms              int64
Type                 str
Price            float64
Method               str
Distance         float64
Postcode         float64
Bedroom2         float64
Bathroom         float64
Car              float64
Landsize         float64
Lattitude        float64
Longtitude       float64
Regionname           str
Propertycount    float64
dtype: object

In [6]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [7]:
print(X.shape)
print(y.shape)

(13580, 14)
(13580,)


In [8]:
categorical_cols = X.select_dtypes(include=['str']).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Categorical: ['Suburb', 'Type', 'Method', 'Regionname']
Numerical: ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'Lattitude', 'Longtitude', 'Propertycount']


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    ElasticNetCV,
    HuberRegressor,
    LassoCV,
    LinearRegression,
    RidgeCV,
    SGDRegressor,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical_features,
            ),
        ]
    )

CANDIDATES = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-2, 6, 33)),
    "Lasso": LassoCV(alphas=100, cv=5, random_state=42, max_iter=2000),
    "Elastic Net": ElasticNetCV(
        l1_ratio=[0.3, 0.5, 0.7, 0.9, 1.0],
        cv=5,
        random_state=42,
        max_iter=2000,
    ),
    "SGD": SGDRegressor(random_state=42, max_iter=2000, tol=1e-3),
    "Huber": HuberRegressor(max_iter=200),
    "KNN": KNeighborsRegressor(n_neighbors=10),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Bagging": BaggingRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=6),
        learning_rate=0.05,
        n_estimators=200,
        random_state=42,
    ),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

models = {}
metrics = {}

for name, estimator in CANDIDATES.items():
    pipe = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("regressor", estimator),
    ])
    pipe.fit(X_train, y_train)
    models[name] = pipe

    preds = pipe.predict(X_test)
    metrics[name] = {
        "R2": r2_score(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds),
    }
    print(f"trained + scored: {name}")


trained + scored: Dummy Regressor


trained + scored: Linear Regression


trained + scored: Ridge


trained + scored: Lasso


trained + scored: Elastic Net


...

trained + scored: AdaBoost


trained + scored: Extra Trees


trained + scored: Random Forest


trained + scored: Gradient Boosting


trained + scored: Hist Gradient Boosting


In [ ]:
import pandas as pd

bundle = {
    "models": models,
    "metrics": metrics,
    "features": X.columns.tolist(),
}

from pathlib import Path

MODEL_PATH = Path("../models/melb-price-model.joblib")
MODEL_PATH.parent.mkdir(exist_ok=True)

joblib.dump(bundle, MODEL_PATH, compress=3)

print(f"Bundle saved: {MODEL_PATH}\n")

report = pd.DataFrame(metrics).T.sort_values("R2", ascending=False)
report_display = pd.DataFrame({
    "R²": report["R2"].map("{:.3f}".format),
    "MAE": report["MAE"].map("${:,.0f}".format),
    "RMSE": report["RMSE"].map("${:,.0f}".format),
})
report_display.index.name = "Algorithm"
report_display

Bundle saved: ../models/melb-price-model.joblib



,R²,MAE,RMSE
Algorithm,,,
Hist Gradient Boosting,0.831,"$164,358","$258,716"
Random Forest,0.817,"$165,554","$269,892"
Bagging,0.813,"$166,708","$272,336"
Extra Trees,0.792,"$173,452","$287,747"
Gradient Boosting,0.775,"$193,656","$298,847"
KNN,0.742,"$202,576","$319,846"
AdaBoost,0.732,"$220,940","$326,088"
Linear Regression,0.676,"$240,826","$358,800"
Ridge,0.675,"$239,679","$359,533"


In [ ]:
from huggingface_hub import HfApi

HfApi().upload_file(
    path_or_fileobj="../models/melb-price-model.joblib",
    path_in_repo="melb-price-model.joblib",
    repo_id="masadullahna/melb-by-na",
    repo_type="model",
    commit_message="Upload trained model bundle",
)